In [1]:
import pandas as pd
from spacy_dish_name_map import *
from similarity_map import *
from config import Baseconfig

/opt/anaconda3/envs/ner_environment/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = Baseconfig()

df_dish = pd.read_csv(config.dish_path)
df = pd.read_csv(config.review_path)
if config.is_test:
    sampled_business_ids = df['business_id'].sample(n=5, random_state=123).tolist()
    df = df[df['business_id'].isin(sampled_business_ids)]
df['text_clean'] = df['text'].apply(clean_text)

In [3]:
df.head()

,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny,text_clean
81,5kRug3bEienrpovtPRVVwg,2012-06-29,BslISEau8Pk2voEPN7kY-g,4,Rokerij is worth the hype. We popped in last w...,review,gcyEUr4DXcbjnGRAWFtfAQ,0,1,0,rokerij is worth the hype we popped in last we...
332,_oQN89kF_d97-cWW0iO8_g,2012-02-17,wa_exNUrfZRlzk8xiqYDxA,2,I read all the good reviews of this restaurant...,review,KpC10_UEJuga43WadHCmYw,0,1,0,i read all the good reviews of this restaurant...
445,6Lo25bdGe3qEdFLGygBsPw,2011-08-21,qbw_6lX5pyM7YBWxHrBbbw,5,"Like others, I found this studio through a Liv...",review,Y_iCmH_z_T5cOy3_8p_pQw,2,2,0,like others i found this studio through a livi...
646,18TUn9oiW0k0yB6lheiOvw,2011-09-09,yj_LYhizV601mC2Y9ql55A,5,Yep -- still my favorite hole-in-the-wall eate...,review,aTi0NVrcPJWbN6jAsJVcAw,0,0,0,yep still my favorite holeinthewall eatery aw...
1792,18TUn9oiW0k0yB6lheiOvw,2011-02-05,CT7j5_WvUtIlMX7PfxJTdQ,5,This is by far the best Chicken Fried Steak ar...,review,ZzxZZksMVxeOGcGfEFbfUw,0,0,0,this is by far the best chicken fried steak ar...


In [4]:
ACTION_VERBS = get_action_verbs(df, 'text_clean', 20)
filtered_df = df_dish[df_dish['last_appeared'] > 2000]
# Extract the names into a list
MENUE_ITEMS = filtered_df['name'].tolist()

In [5]:
df['ngram_map'] = df['text_clean'].apply(lambda x: ngram_edit_search(x, MENUE_ITEMS))
df['spacy_map'] = df['text_clean'].apply(lambda x: spacy_map(x, MENUE_ITEMS))
df['st_map'] = df['text_clean'].apply(lambda x: st_map(x, MENUE_ITEMS))

In [6]:
print(df['text'][81])

Rokerij is worth the hype. We popped in last week for a late bite and left super happy. We scored a few seats at the bar, the place was SUPER crowded (which is a good sign on a Wednesday). I started with a beer called Banana Bread, which was awesome. Wifey had a glass of wine, but I can't remember what it was. We couldn't decide on what we wnated so we ended up splitting the entree of Blackened Salmon with Apple/Chipotle chutney and some potatoes. We also got three of the small Plates: mini reuben, crispy calamari and cilantro rubbed grilled shrimp. All the food was awesome. The salmon was cooked perfectly, although i think I would have liked the chutney on the side, but that's small fries. The calamari was cooked perfectly, the shrimp were awesome (although the bed of rice it came on was meh), and the mini reubens were great as well. I was super excited to try it, and it delivered. Can't wait to go back!


In [7]:
print(df['ngram_map'][81])

['Potatoes', 'Entrees', 'Shrimp', 'Bananas']


In [8]:
print(df['spacy_map'][81])

['Cotuits', 'Cherries', 'Apfelschorle', 'Drambuie', 'Blattsalat', 'Burger', 'Fraise', 'Sauternes', 'Dessert', 'Roederer', 'Caffe', 'Glühwein', 'Fruit', 'Underberg', 'Chiffonade', 'Potatoes', 'Combination', 'Orangensaft', 'Vichyssoise', 'Coffee', 'Decafeine', 'Camembert', 'Shrimp']


In [9]:
print(df['st_map'][81])

['Bar Snacks', 'Salad', 'Sides', 'Cheese Plate', 'Guinness', 'Entrees', 'Bananas', 'Potatoes', 'Wild Salmon - with horseradish crust, cabbage and Riesling', 'Crispy Yellowfin Tuna - with roasted asparagus, greenmarket spring beans, warm tomato vinaigrette and shaved bottarga', 'Calvados', 'Apples', "Healy's Oatmeal Muffins or Corn Bread", 'French fries', 'Shrimp', 'Wine of the Day']


In [10]:
df.head()

,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny,text_clean,ngram_map,spacy_map,st_map
81,5kRug3bEienrpovtPRVVwg,2012-06-29,BslISEau8Pk2voEPN7kY-g,4,Rokerij is worth the hype. We popped in last w...,review,gcyEUr4DXcbjnGRAWFtfAQ,0,1,0,rokerij is worth the hype we popped in last we...,"[Potatoes, Entrees, Shrimp, Bananas]","[Cotuits, Cherries, Apfelschorle, Drambuie, Bl...","[Bar Snacks, Salad, Sides, Cheese Plate, Guinn..."
332,_oQN89kF_d97-cWW0iO8_g,2012-02-17,wa_exNUrfZRlzk8xiqYDxA,2,I read all the good reviews of this restaurant...,review,KpC10_UEJuga43WadHCmYw,0,1,0,i read all the good reviews of this restaurant...,[],"[Milanaise, peaches, Zwetschgenwasser, Cappucc...","[Salad, Mexican Vegetable Burritos, Tortillas ..."
445,6Lo25bdGe3qEdFLGygBsPw,2011-08-21,qbw_6lX5pyM7YBWxHrBbbw,5,"Like others, I found this studio through a Liv...",review,Y_iCmH_z_T5cOy3_8p_pQw,2,2,0,like others i found this studio through a livi...,[],"[Sauternes, Potatoes, Saran, CAFE]",[]
646,18TUn9oiW0k0yB6lheiOvw,2011-09-09,yj_LYhizV601mC2Y9ql55A,5,Yep -- still my favorite hole-in-the-wall eate...,review,aTi0NVrcPJWbN6jAsJVcAw,0,0,0,yep still my favorite holeinthewall eatery aw...,[],"[Jägermeister, Chiffonade, Chardonnay, Fraise,...","[Bar Snacks, Salad, Cheeseburger, Club sandwic..."
1792,18TUn9oiW0k0yB6lheiOvw,2011-02-05,CT7j5_WvUtIlMX7PfxJTdQ,5,This is by far the best Chicken Fried Steak ar...,review,ZzxZZksMVxeOGcGfEFbfUw,0,0,0,this is by far the best chicken fried steak ar...,"[Salad, Potatoes]","[Salad, Steinhäger, Blattsalat, Tomatensaft, C...","[Sliced Chicken, Salad, Biscuit Tortoni, Frenc..."


In [11]:
# test_df = apply_spacy_to_df(df, ACTION_VERBS, MENUE_ITEMS, True, 3)

In [12]:
def frequency_filter(df, target_col, frequency_threshold=0.1):
    # Count total observations per business_id
    n_obs = df.groupby('business_id').size().rename('n_obs')
    
    # Explode and count unique observations per string
    exploded = df.explode(target_col).reset_index()
    str_counts = exploded.groupby(['business_id', target_col])['index'].nunique().reset_index(name='count')
    
    # Merge with total counts and calculate frequency
    merged = str_counts.merge(n_obs, on='business_id', how='right')
    merged['count'] = merged['count'].fillna(0)
    merged['keep'] = merged['count'] > (merged['n_obs'] * frequency_threshold)
    
    # Create filtered lists and ensure all business_ids are included
    filtered = (
        merged[merged['keep']]
        .groupby('business_id')[target_col]
        .agg(list)
        .reindex(n_obs.index)
        .fillna({target_col: []})
        .reset_index()
    )
    
    return filtered

In [13]:
result_df = frequency_filter(df,'st_map', 0.05)

In [14]:
print(result_df.st_map[0])

['Baked Potatoes', 'Bar Snacks', 'Biscuit Tortoni', 'Burger', 'Cheese Plate', 'Cheeseburger', 'Club sandwich', 'Cocktail Sauce', 'Combination', 'Dessert', 'French Dressing', 'Half-Sandwich (Curried Chicken Salad) and Soup of the Day', 'Hanger Steak', 'Potatoes', 'Salad', 'Sliced Chicken']
